

## 1. Environment Setup


In [ ]:
print(" Installing required packages for X-CheckList")
print("This may take several minutes in Google Colab.")

import sys, subprocess, importlib, platform

required_packages = [
    "datasets", "transformers", "accelerate", "torch", "pandas", "numpy",
    "scikit-learn", "matplotlib", "seaborn", "scipy", "tqdm", "captum", "shap"
]

IMPORT_NAMES = {
    "scikit-learn": "sklearn",
    "accelerate": "accelerate",
}

def install_package(package_name):
    import_name = IMPORT_NAMES.get(package_name, package_name.replace("-", "_"))
    try:
        importlib.import_module(import_name)
        print(f"Already available: {package_name}")
    except Exception:
        print(f" Installing: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        print(f"Installed: {package_name}")

for pkg in required_packages:
    install_package(pkg)

print("Package setup completed")
print("Python executable:", sys.executable)
print("Python version:", platform.python_version())


In [ ]:
print("Importing libraries and selecting device...")

import os, re, json, math, random, warnings, gc, traceback
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from scipy.stats import wilcoxon
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

try:
    import shap
    SHAP_AVAILABLE = True
    print(" SHAP imported")
except Exception as e:
    SHAP_AVAILABLE = False
    print("SHAP unavailable:", repr(e))

try:
    from captum.attr import IntegratedGradients
    CAPTUM_AVAILABLE = True
    print("Captum imported")
except Exception as e:
    CAPTUM_AVAILABLE = False
    print("Captum unavailable:", repr(e))

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Imports completed")
print("Device:", device)
print("PyTorch:", torch.__version__)



## 2. Configuration




In [ ]:
print("Creating configuration...")

RUN_MODE = "full"  # options: "debug" or "full"

CONFIG_PRESETS = {
    "debug": {
        "datasets": ["sst2", "imdb", "mnli", "snli", "qqp", "mrpc"],
        "models": ["distilbert", "bert", "roberta"],
        "sample_size_per_dataset": 40,
        "template_cases_per_capability": 20,
        "max_explain_cases_per_combo": 50,
        "explanation_methods": ["occlusion", "gradient"],
        "use_shap": False,
        "use_integrated_gradients": False,
        "fast_mode": True,
    },
    "full": {
        "datasets": ["sst2", "imdb", "mnli", "snli", "qqp", "mrpc"],
        "models": ["distilbert", "bert", "roberta"],
        "sample_size_per_dataset": 500,
        "template_cases_per_capability": 300,
        "max_explain_cases_per_combo": 300,  # set None to explain every failed case
        "explanation_methods": ["occlusion", "gradient", "integrated_gradients"],
        "use_shap": False,  # SHAP is slow; enable only for final sampled evidence if needed
        "use_integrated_gradients": True,
        "fast_mode": False,
    }
}

CONFIG = {
    **CONFIG_PRESETS[RUN_MODE],
    "output_dir": "/content/outputs" if "google.colab" in sys.modules else "outputs_q1",
    "figures_dir_name": "figures",
    "random_seed": 42,
    "batch_size": 16,
    "max_length": 128,
    "counterfactual_cs_threshold": 0.20,
    "am_high_threshold": 0.60,
    "bootstrap_samples": 1000,
    "human_eval_sample_size": 75,
    "human_eval_methods": [
    "X-CheckList",
    "Standard CheckList",
    "CheckList + Attribution",
    "CheckList + Counterfactual"
    ],
    "human_eval_annotator_ids": ["A1", "A2"],
    "human_eval_strata_cols": ["dataset", "model", "capability"],
    "save_to_google_drive": True,
    "google_drive_output_dir": "/content/drive/MyDrive/X_CheckList_outputs",
}

random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])
torch.manual_seed(CONFIG["random_seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["random_seed"])

output_dir = Path(CONFIG["output_dir"])
figures_dir = output_dir / CONFIG["figures_dir_name"]
output_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

generated_files = []

print("CONFIG created")
print(json.dumps(CONFIG, indent=2))
print("Output directory:", output_dir.resolve())
print("Figures directory:", figures_dir.resolve())




## 3. Dataset Loading Utilities


In [ ]:
print("Defining dataset loading utilities for SST-2, IMDB, MNLI, SNLI, QQP, MRPC...")

DATASET_SPECS = {
    "sst2": {"hf_name": "nyu-mll/glue", "subset": "sst2", "split": "validation", "task": "sentiment", "text_cols": ["sentence"], "label_col": "label"},
    "imdb": {"hf_name": "imdb", "subset": None, "split": "test", "task": "sentiment", "text_cols": ["text"], "label_col": "label"},
    "mnli": {"hf_name": "nyu-mll/glue", "subset": "mnli", "split": "validation_matched", "task": "nli", "text_cols": ["premise", "hypothesis"], "label_col": "label"},
    "snli": {"hf_name": "snli", "subset": None, "split": "validation", "task": "nli", "text_cols": ["premise", "hypothesis"], "label_col": "label"},
    "qqp": {"hf_name": "nyu-mll/glue", "subset": "qqp", "split": "validation", "task": "paraphrase", "text_cols": ["question1", "question2"], "label_col": "label"},
    "mrpc": {"hf_name": "nyu-mll/glue", "subset": "mrpc", "split": "validation", "task": "paraphrase", "text_cols": ["sentence1", "sentence2"], "label_col": "label"},
}

LABEL_MAPS = {
    "sentiment": {0: "NEGATIVE", 1: "POSITIVE"},
    "nli": {0: "ENTAILMENT", 1: "NEUTRAL", 2: "CONTRADICTION", -1: "INVALID"},
    "paraphrase": {0: "NOT_EQUIVALENT", 1: "EQUIVALENT"},
}

def load_one_dataset(dataset_key, sample_size=None, seed=42):
    spec = DATASET_SPECS[dataset_key]
    print(f"Loading {dataset_key.upper()} from Hugging Face: {spec['hf_name']} {spec['subset'] or ''} / {spec['split']}")
    try:
        if spec["subset"]:
            ds = load_dataset(spec["hf_name"], spec["subset"])
        else:
            ds = load_dataset(spec["hf_name"])
        split = ds[spec["split"]]
        df = pd.DataFrame(split)
        if dataset_key == "snli":
            df = df[df[spec["label_col"]] != -1].copy()
        original_rows = len(df)
        if sample_size is not None and sample_size < len(df):
            df = df.sample(n=sample_size, random_state=seed).reset_index(drop=True)
        else:
            df = df.reset_index(drop=True)
        df["dataset"] = dataset_key
        df["task"] = spec["task"]
        print(f"Loaded {dataset_key}: original={original_rows}, returned={len(df)}")
        print("Columns:", list(df.columns))
        print("Label distribution:")
        print(df[spec["label_col"]].value_counts().sort_index())
        display(df.head(3))
        return df
    except Exception as e:
        print(f"Failed to load {dataset_key}: {repr(e)}")
        return pd.DataFrame()

loaded_datasets = {}
for dataset_key in CONFIG["datasets"]:
    df = load_one_dataset(dataset_key, sample_size=CONFIG["sample_size_per_dataset"], seed=CONFIG["random_seed"])
    if len(df) > 0:
        loaded_datasets[dataset_key] = df

print("Dataset loading completed")
print("Loaded datasets:", list(loaded_datasets.keys()))
for k, df in loaded_datasets.items():
    print(k, df.shape)


if loaded_datasets:
    sampled_input_instances = pd.concat(loaded_datasets.values(), ignore_index=True)
    sampled_inputs_path = output_dir / "sampled_input_instances.csv"
    sampled_input_instances.to_csv(sampled_inputs_path, index=False)
    generated_files.append(str(sampled_inputs_path))
    print("Saved exact sampled input instances:", sampled_inputs_path)
    display(sampled_input_instances.head())




## 4. Behavioral Test Case Construction

Each test case contains input, expected label, capability, relation type, and optional counterfactual pair.


In [ ]:
print(" Defining behavioral test construction utilities...")

POS_WORDS = ["good", "great", "excellent", "amazing", "wonderful", "pleasant", "enjoyable", "fantastic", "positive", "loved"]
NEG_WORDS = ["bad", "terrible", "awful", "poor", "boring", "horrible", "disappointing", "negative", "hated", "worst"]
NAMES = [("John", "Mary"), ("Alice", "Robert"), ("David", "Sarah"), ("Michael", "Emma")]


def canonical_label(task, raw_label):
    return LABEL_MAPS[task].get(int(raw_label), str(raw_label).upper())


def make_test_case(test_id, dataset, task, input_text, expected_label, capability, relation,
                   counterfactual_text=None, counterfactual_expected_label=None,
                   sentence1=None, sentence2=None, cf_sentence1=None, cf_sentence2=None,
                   expected_cues=None, misleading_cues=None, source="template", metadata=None):
    return {
        "test_id": test_id,
        "dataset": dataset,
        "task": task,
        "input_text": input_text,
        "sentence1": sentence1,
        "sentence2": sentence2,
        "expected_label": expected_label,
        "capability": capability,
        "relation": relation,  # directional or invariance
        "counterfactual_text": counterfactual_text,
        "cf_sentence1": cf_sentence1,
        "cf_sentence2": cf_sentence2,
        "counterfactual_expected_label": counterfactual_expected_label if counterfactual_expected_label is not None else expected_label,
        "expected_cues": expected_cues or [],
        "misleading_cues": misleading_cues or [],
        "source": source,
        "metadata": json.dumps(metadata or {}, ensure_ascii=False),
    }


def typo_perturb(text):
    replacements = [("good", "goood"), ("great", "greeat"), ("bad", "baad"), ("movie", "moovie"),
                    ("film", "fiilm"), ("excellent", "exceellent"), ("terrible", "terrrible")]
    for old, new in replacements:
        if re.search(r"\b" + re.escape(old) + r"\b", text, flags=re.I):
            return re.sub(r"\b" + re.escape(old) + r"\b", new, text, count=1, flags=re.I), [new]
    words = text.split()
    if not words:
        return text, []
    idx = min(1, len(words)-1)
    words[idx] = words[idx] + words[idx][-1]
    return " ".join(words), [words[idx]]


def build_sentiment_tests(dataset_key, df, limit_per_capability=100):
    rows = []
    # Template negation/directional tests
    i = 0
    for w in POS_WORDS:
        if i >= limit_per_capability: break
        orig = f"The movie was {w}."
        cf = f"The movie was not {w}."
        rows.append(make_test_case(f"{dataset_key}_neg_{i:04d}", dataset_key, "sentiment", cf, "NEGATIVE", "negation", "directional",
                                  counterfactual_text=orig, counterfactual_expected_label="POSITIVE",
                                  expected_cues=["not"], misleading_cues=[w], source="template"))
        i += 1
    for w in NEG_WORDS:
        if i >= limit_per_capability: break
        orig = f"The movie was {w}."
        cf = f"The movie was not {w}."
        rows.append(make_test_case(f"{dataset_key}_neg_{i:04d}", dataset_key, "sentiment", cf, "POSITIVE", "negation", "directional",
                                  counterfactual_text=orig, counterfactual_expected_label="NEGATIVE",
                                  expected_cues=["not"], misleading_cues=[w], source="template"))
        i += 1

    # Temporal sentiment templates
    templates = [
        ("I used to hate this movie, but now I like it.", "POSITIVE", "I used to like this movie, but now I hate it.", "NEGATIVE", ["but", "now", "like"], ["hate"]),
        ("At first the film was boring, but finally it became excellent.", "POSITIVE", "At first the film was excellent, but finally it became boring.", "NEGATIVE", ["but", "finally", "excellent"], ["boring"]),
        ("Earlier the service was bad, but now it is wonderful.", "POSITIVE", "Earlier the service was wonderful, but now it is bad.", "NEGATIVE", ["but", "now", "wonderful"], ["bad"]),
        ("I expected it to be great, but it ended terribly.", "NEGATIVE", "I expected it to be terrible, but it ended greatly.", "POSITIVE", ["but", "ended", "terribly"], ["great"]),
    ]
    for j in range(limit_per_capability):
        t = templates[j % len(templates)]
        rows.append(make_test_case(f"{dataset_key}_temp_{j:04d}", dataset_key, "sentiment", t[0], t[1], "temporal_sentiment", "directional",
                                  counterfactual_text=t[2], counterfactual_expected_label=t[3], expected_cues=t[4], misleading_cues=t[5], source="template"))

    # Entity invariance templates
    for j in range(limit_per_capability):
        n1, n2 = NAMES[j % len(NAMES)]
        sent = f"{n1} said the service was excellent."
        cf = f"{n2} said the service was excellent."
        rows.append(make_test_case(f"{dataset_key}_entity_{j:04d}", dataset_key, "sentiment", sent, "POSITIVE", "entity_invariance", "invariance",
                                  counterfactual_text=cf, counterfactual_expected_label="POSITIVE", expected_cues=["excellent"], misleading_cues=[n1, n2], source="template"))

    # Robustness from dataset samples + templates
    if len(df) > 0:
        spec = DATASET_SPECS[dataset_key]
        text_col = spec["text_cols"][0]
        for j, (_, r) in enumerate(df.head(limit_per_capability).iterrows()):
            text = str(r[text_col])[:500]
            label = canonical_label("sentiment", r[spec["label_col"]])
            cf, perturb = typo_perturb(text)
            rows.append(make_test_case(f"{dataset_key}_robust_{j:04d}", dataset_key, "sentiment", text, label, "robustness", "invariance",
                                      counterfactual_text=cf, counterfactual_expected_label=label, expected_cues=[], misleading_cues=perturb, source="benchmark"))
    return rows


def build_nli_tests(dataset_key, df, limit_per_capability=100):
    rows=[]
    # Lexical overlap / role reversal
    templates = [
        ("A dog is chasing a man.", "A man is chasing a dog.", "NEUTRAL"),
        ("The boy is helping the girl.", "The girl is helping the boy.", "NEUTRAL"),
        ("A teacher is following a student.", "A student is following a teacher.", "NEUTRAL"),
        ("The cat is above the box.", "The box is above the cat.", "NEUTRAL"),
    ]
    for j in range(limit_per_capability):
        p, h, lab = templates[j % len(templates)]
        rows.append(make_test_case(f"{dataset_key}_lex_{j:04d}", dataset_key, "nli", p + " [SEP] " + h, lab, "lexical_overlap", "directional",
                                  sentence1=p, sentence2=h, cf_sentence1=p, cf_sentence2=p, counterfactual_expected_label="ENTAILMENT",
                                  expected_cues=["chasing", "helping", "following", "above"], misleading_cues=list(set(p.lower().split()) & set(h.lower().split())), source="template"))
    # Contradiction
    contradiction_templates = [
        ("A person is playing guitar.", "No person is playing guitar."),
        ("The child is sleeping.", "The child is not sleeping."),
        ("A man is outdoors.", "A man is not outdoors."),
        ("Two people are sitting.", "Nobody is sitting."),
    ]
    for j in range(limit_per_capability):
        p,h = contradiction_templates[j % len(contradiction_templates)]
        rows.append(make_test_case(f"{dataset_key}_contr_{j:04d}", dataset_key, "nli", p + " [SEP] " + h, "CONTRADICTION", "contradiction", "directional",
                                  sentence1=p, sentence2=h, cf_sentence1=p, cf_sentence2=p, counterfactual_expected_label="ENTAILMENT",
                                  expected_cues=["no", "not", "nobody"], misleading_cues=["playing", "sleeping", "outdoors", "sitting"], source="template"))
    # Semantic role sensitivity from benchmark where available
    if len(df) > 0:
        spec = DATASET_SPECS[dataset_key]
        c1, c2 = spec["text_cols"]
        for j, (_, r) in enumerate(df.head(limit_per_capability).iterrows()):
            p, h = str(r[c1]), str(r[c2])
            lab = canonical_label("nli", r[spec["label_col"]])
            rows.append(make_test_case(f"{dataset_key}_bench_{j:04d}", dataset_key, "nli", p + " [SEP] " + h, lab, "semantic_role_sensitivity", "invariance",
                                      sentence1=p, sentence2=h, cf_sentence1=p, cf_sentence2=h, counterfactual_expected_label=lab,
                                      expected_cues=[], misleading_cues=[], source="benchmark"))
    return rows


def build_paraphrase_tests(dataset_key, df, limit_per_capability=100):
    rows=[]
    # Entity invariance and equivalence
    for j in range(limit_per_capability):
        n1,n2 = NAMES[j % len(NAMES)]
        q1 = f"Did {n1} enjoy the movie?"
        q2 = f"Did {n1} like the film?"
        cf1 = f"Did {n2} enjoy the movie?"
        cf2 = f"Did {n2} like the film?"
        rows.append(make_test_case(f"{dataset_key}_para_ent_{j:04d}", dataset_key, "paraphrase", q1 + " [SEP] " + q2, "EQUIVALENT", "entity_invariance", "invariance",
                                  sentence1=q1, sentence2=q2, cf_sentence1=cf1, cf_sentence2=cf2, counterfactual_expected_label="EQUIVALENT",
                                  expected_cues=["enjoy", "like"], misleading_cues=[n1,n2], source="template"))
    # Word order sensitivity / non-equivalence
    templates = [
        ("Can a dog chase a cat?", "Can a cat chase a dog?"),
        ("Is Paris larger than Lyon?", "Is Lyon larger than Paris?"),
        ("Did the student teach the professor?", "Did the professor teach the student?"),
    ]
    for j in range(limit_per_capability):
        q1,q2 = templates[j % len(templates)]
        rows.append(make_test_case(f"{dataset_key}_word_{j:04d}", dataset_key, "paraphrase", q1 + " [SEP] " + q2, "NOT_EQUIVALENT", "word_order_sensitivity", "directional",
                                  sentence1=q1, sentence2=q2, cf_sentence1=q1, cf_sentence2=q1, counterfactual_expected_label="EQUIVALENT",
                                  expected_cues=["chase", "larger", "teach"], misleading_cues=list(set(q1.lower().split()) & set(q2.lower().split())), source="template"))
    # Benchmark-derived semantic equivalence/non-equivalence
    if len(df) > 0:
        spec = DATASET_SPECS[dataset_key]
        c1,c2 = spec["text_cols"]
        for j, (_, r) in enumerate(df.head(limit_per_capability).iterrows()):
            q1,q2 = str(r[c1]), str(r[c2])
            lab = canonical_label("paraphrase", r[spec["label_col"]])
            rows.append(make_test_case(f"{dataset_key}_bench_{j:04d}", dataset_key, "paraphrase", q1 + " [SEP] " + q2, lab, "semantic_equivalence", "invariance",
                                      sentence1=q1, sentence2=q2, cf_sentence1=q1, cf_sentence2=q2, counterfactual_expected_label=lab,
                                      expected_cues=[], misleading_cues=[], source="benchmark"))
    return rows

print("Test construction utilities ready")


In [ ]:
print("Building behavioral test cases for all configured datasets...")

all_tests = []
limit = CONFIG["template_cases_per_capability"]
for dataset_key, df in loaded_datasets.items():
    task = DATASET_SPECS[dataset_key]["task"]
    if task == "sentiment":
        all_tests.extend(build_sentiment_tests(dataset_key, df, limit_per_capability=limit))
    elif task == "nli":
        all_tests.extend(build_nli_tests(dataset_key, df, limit_per_capability=limit))
    elif task == "paraphrase":
        all_tests.extend(build_paraphrase_tests(dataset_key, df, limit_per_capability=limit))

test_cases_df = pd.DataFrame(all_tests).drop_duplicates(subset=["dataset", "input_text", "sentence1", "sentence2", "capability"]).reset_index(drop=True)

print("Behavioral test cases generated")
print("Total test cases:", len(test_cases_df))
print("By dataset:")
print(test_cases_df["dataset"].value_counts())
print("By capability:")
print(test_cases_df["capability"].value_counts())
print("By relation:")
print(test_cases_df["relation"].value_counts())
display(test_cases_df.head(10))

test_cases_path = output_dir / "xchecklist_test_cases.csv"
test_cases_df.to_csv(test_cases_path, index=False)
generated_files.append(str(test_cases_path))
print("Saved:", test_cases_path)



## 5. Model Registry and Prediction Wrapper

The model is loaded per **dataset × model** pair, not globally. This avoids using an SST-2 sentiment model for MNLI/QQP.


In [ ]:
print(" Defining dataset-wise model registry...")

MODEL_REGISTRY = {
    "sst2": {
        "distilbert": "distilbert-base-uncased-finetuned-sst-2-english",
        "bert": "textattack/bert-base-uncased-SST-2",
        "roberta": "textattack/roberta-base-SST-2",

    },
    "imdb": {
        "distilbert": "lvwerra/distilbert-imdb",
        "bert": "textattack/bert-base-uncased-imdb",
        "roberta": "textattack/roberta-base-imdb",

    },
    "mnli": {
        "distilbert": "typeform/distilbert-base-uncased-mnli",
        "bert": "textattack/bert-base-uncased-MNLI",
        "roberta": "roberta-large-mnli",

    },
    "snli": {
        "distilbert": "typeform/distilbert-base-uncased-mnli",
        "bert": "textattack/bert-base-uncased-MNLI",
        "roberta": "roberta-large-mnli",

    },
    "qqp": {
        "distilbert": "textattack/distilbert-base-uncased-QQP",
        "bert": "textattack/bert-base-uncased-QQP",


    },
    "mrpc": {
        "distilbert": "textattack/distilbert-base-uncased-MRPC",
        "bert": "textattack/bert-base-uncased-MRPC",
        "roberta": "textattack/roberta-base-MRPC",

    },
}

TASK_LABELS = {
    "sentiment": ["NEGATIVE", "POSITIVE"],
    "nli": ["ENTAILMENT", "NEUTRAL", "CONTRADICTION"],
    "paraphrase": ["NOT_EQUIVALENT", "EQUIVALENT"],
}


MODEL_LABEL_ORDER_OVERRIDES = {
    "roberta-large-mnli": ["CONTRADICTION", "NEUTRAL", "ENTAILMENT"],
    "typeform/distilbert-base-uncased-mnli": ["CONTRADICTION", "NEUTRAL", "ENTAILMENT"],
    "textattack/bert-base-uncased-MNLI": ["ENTAILMENT", "NEUTRAL", "CONTRADICTION"],
}


def normalize_model_label(raw_label, task, label_id=None, model_name=None):
    s = str(raw_label).upper().replace("-", "_").replace(" ", "_")

    if task == "sentiment":
        if "NEG" in s or s in ["0", "LABEL_0"]: return "NEGATIVE"
        if "POS" in s or s in ["1", "LABEL_1"]: return "POSITIVE"
        return TASK_LABELS[task][int(label_id)] if label_id is not None and int(label_id) < 2 else s

    if task == "nli":
        if "ENTAIL" in s and "NOT_ENTAIL" not in s: return "ENTAILMENT"
        if "NEUT" in s: return "NEUTRAL"
        if "CONTR" in s or "NOT_ENTAIL" in s: return "CONTRADICTION"
        if label_id is not None and model_name in MODEL_LABEL_ORDER_OVERRIDES:
            order = MODEL_LABEL_ORDER_OVERRIDES[model_name]
            if int(label_id) < len(order):
                return order[int(label_id)]
        # GLUE canonical fallback when no checkpoint-specific order is known.
        return {0: "ENTAILMENT", 1: "NEUTRAL", 2: "CONTRADICTION"}.get(int(label_id), s)

    if task == "paraphrase":
        if "NOT" in s or "LABEL_0" in s or s == "0": return "NOT_EQUIVALENT"
        if "EQUIV" in s or "PARAPHRASE" in s or "LABEL_1" in s or s == "1": return "EQUIVALENT"
        return {0: "NOT_EQUIVALENT", 1: "EQUIVALENT"}.get(int(label_id), s)

    return s

class TargetModelWrapper:
    def __init__(self, model_key, dataset_key, model_name, task, device=device, max_length=128):
        self.model_key = model_key
        self.dataset_key = dataset_key
        self.model_name = model_name
        self.task = task
        self.device = device
        self.max_length = max_length
        print(f"Loading {model_key} for {dataset_key}: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()
        self.id2label = {int(k): v for k, v in self.model.config.id2label.items()} if hasattr(self.model.config, "id2label") else {}
        self.label2id = getattr(self.model.config, "label2id", {}) if hasattr(self.model.config, "label2id") else {}
        print("id2label:", self.id2label)

    def probabilities_by_normalized_label(self, prob):
        label_probs = defaultdict(float)
        for label_id, p in enumerate(prob):
            raw_label = self.id2label.get(label_id, f"LABEL_{label_id}")
            norm_label = normalize_model_label(raw_label, self.task, label_id, self.model_name)
            label_probs[norm_label] += float(p)
        return dict(label_probs)

    def predict_batch(self, rows):
        texts1, texts2 = [], []
        for row in rows:
            if pd.notna(row.get("sentence1", np.nan)) and pd.notna(row.get("sentence2", np.nan)) and str(row.get("sentence2")) != "nan":
                texts1.append(str(row["sentence1"]))
                texts2.append(str(row["sentence2"]))
            else:
                texts1.append(str(row["input_text"]))
                texts2.append(None)
        if any(x is not None for x in texts2):
            enc = self.tokenizer(texts1, texts2, padding=True, truncation=True, max_length=self.max_length, return_tensors="pt")
        else:
            enc = self.tokenizer(texts1, padding=True, truncation=True, max_length=self.max_length, return_tensors="pt")
        enc = {k: v.to(self.device) for k, v in enc.items()}
        with torch.no_grad():
            logits = self.model(**enc).logits
            probs = F.softmax(logits, dim=-1).detach().cpu().numpy()
        outs=[]
        for prob in probs:
            pred_id = int(np.argmax(prob))
            raw_label = self.id2label.get(pred_id, f"LABEL_{pred_id}")
            pred_label = normalize_model_label(raw_label, self.task, pred_id, self.model_name)
            probs_by_label = self.probabilities_by_normalized_label(prob)
            outs.append({
                "predicted_label": pred_label,
                "predicted_label_id": pred_id,
                "confidence": float(prob[pred_id]),
                "probabilities": prob.tolist(),
                "probabilities_by_label": probs_by_label,
                "raw_predicted_label": raw_label,
            })
        return outs

    def predict_one(self, sentence1=None, sentence2=None, input_text=None):
        row = {"sentence1": sentence1, "sentence2": sentence2, "input_text": input_text or sentence1}
        return self.predict_batch([row])[0]

    def cleanup(self):
        try:
            self.model.cpu()
        except Exception:
            pass
        del self.model
        del self.tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("Model registry and wrapper ready")
print(json.dumps(MODEL_REGISTRY, indent=2))




## 6. Failure Detection and Counterfactual Sensitivity


In [ ]:
print("Defining prediction, failure detection, and CS utilities...")

def get_counterfactual_row(row):
    cf_row = row.copy()
    if pd.notna(row.get("cf_sentence1", np.nan)) and str(row.get("cf_sentence1")) != "nan":
        cf_row["sentence1"] = row.get("cf_sentence1")
        cf_row["sentence2"] = row.get("cf_sentence2")
        cf_row["input_text"] = str(row.get("cf_sentence1")) + " [SEP] " + str(row.get("cf_sentence2"))
    elif pd.notna(row.get("counterfactual_text", np.nan)) and str(row.get("counterfactual_text")) != "nan":
        cf_row["input_text"] = row.get("counterfactual_text")
        cf_row["sentence1"] = np.nan
        cf_row["sentence2"] = np.nan
    return cf_row


def probability_for_label(pred_output, expected_label, task):
    """Return probability for a normalized task label.

    Uses wrapper-generated probabilities_by_label so CS is not corrupted by checkpoint-specific
    label order differences such as RoBERTa-MNLI using contradiction/neutral/entailment order.
    """
    probs_by_label = pred_output.get("probabilities_by_label", {}) or {}
    if expected_label in probs_by_label:
        return float(probs_by_label[expected_label])

    # Safe fallback for older outputs.
    labels = TASK_LABELS[task]
    probs = pred_output.get("probabilities", [])
    if expected_label in labels:
        idx = labels.index(expected_label)
        if idx < len(probs):
            return float(probs[idx])
    return float(pred_output.get("confidence", 0.0))


def counterfactual_sensitivity(orig_pred, cf_pred, row):
    task = row["task"]
    expected_label = row["expected_label"]
    p_orig = probability_for_label(orig_pred, expected_label, task)
    p_cf = probability_for_label(cf_pred, expected_label, task)
    return abs(p_orig - p_cf)


def run_predictions_for_combo(dataset_key, model_key, wrapper, tests_subset):
    print(f"Running predictions: dataset={dataset_key}, model={model_key}, tests={len(tests_subset)}")
    rows = tests_subset.to_dict("records")
    result_rows=[]
    batch_size = CONFIG["batch_size"]
    for start in tqdm(range(0, len(rows), batch_size), desc=f"{dataset_key}-{model_key} predictions"):
        batch = rows[start:start+batch_size]
        orig_preds = wrapper.predict_batch(batch)
        cf_batch = [get_counterfactual_row(pd.Series(r)).to_dict() for r in batch]
        cf_preds = wrapper.predict_batch(cf_batch)
        for r, op, cp in zip(batch, orig_preds, cf_preds):
            fail = int(op["predicted_label"] != r["expected_label"])
            cf_expected = r.get("counterfactual_expected_label", r["expected_label"])
            cf_fail = int(cp["predicted_label"] != cf_expected)
            cs = counterfactual_sensitivity(op, cp, r)
            result_rows.append({**r,
                "model": model_key,
                "model_checkpoint": wrapper.model_name,
                "predicted_label": op["predicted_label"],
                "predicted_label_id": op["predicted_label_id"],
                "raw_predicted_label": op["raw_predicted_label"],
                "confidence": op["confidence"],
                "probabilities_json": json.dumps(op["probabilities"]),
                "probabilities_by_label_json": json.dumps(op.get("probabilities_by_label", {})),
                "fail_status": fail,
                "cf_predicted_label": cp["predicted_label"],
                "cf_confidence": cp["confidence"],
                "cf_probabilities_json": json.dumps(cp["probabilities"]),
                "cf_probabilities_by_label_json": json.dumps(cp.get("probabilities_by_label", {})),
                "cf_fail_status": cf_fail,
                "CS": cs,
            })
    return pd.DataFrame(result_rows)

print(" Prediction utilities ready")




## 7. Explanation, Attribution Mismatch, EVFS, and Failure Reason Inference


In [ ]:
print("Defining explanation and diagnostic scoring utilities...")

TOKEN_RE = re.compile(r"[A-Za-z0-9_']+")

def simple_tokens(text):
    return TOKEN_RE.findall(str(text).lower())


def row_text_for_explanation(row):
    if pd.notna(row.get("sentence1", np.nan)) and pd.notna(row.get("sentence2", np.nan)) and str(row.get("sentence2")) != "nan":
        return str(row["sentence1"]) + " [SEP] " + str(row["sentence2"])
    return str(row.get("input_text", ""))


def occlusion_attribution(wrapper, row, predicted_label):
    text = row_text_for_explanation(row)
    words = text.split()
    if len(words) == 0:
        return {}
    base_pred = wrapper.predict_one(sentence1=row.get("sentence1") if pd.notna(row.get("sentence1", np.nan)) else None,
                                    sentence2=row.get("sentence2") if pd.notna(row.get("sentence2", np.nan)) else None,
                                    input_text=row.get("input_text"))
    base_conf = base_pred["confidence"]
    scores={}
    max_words = min(len(words), 80 if CONFIG["fast_mode"] else 160)
    for i in range(max_words):
        masked_words = words.copy()
        masked_words[i] = wrapper.tokenizer.mask_token if wrapper.tokenizer.mask_token else "[MASK]"
        masked_text = " ".join(masked_words)
        pred = wrapper.predict_one(input_text=masked_text)
        scores[words[i].lower().strip(".,!?;:'\"")] = max(0.0, base_conf - pred["confidence"])
    total = sum(scores.values())
    if total > 0:
        scores = {k: float(v/total) for k,v in scores.items()}
    return scores


def gradient_attribution(wrapper, row, predicted_label_id):
    # Lightweight white-box gradient norm over input embeddings.
    try:
        if pd.notna(row.get("sentence1", np.nan)) and pd.notna(row.get("sentence2", np.nan)) and str(row.get("sentence2")) != "nan":
            enc = wrapper.tokenizer(str(row["sentence1"]), str(row["sentence2"]), truncation=True, max_length=wrapper.max_length, return_tensors="pt")
        else:
            enc = wrapper.tokenizer(str(row.get("input_text", "")), truncation=True, max_length=wrapper.max_length, return_tensors="pt")
        enc = {k: v.to(wrapper.device) for k,v in enc.items()}
        input_ids = enc["input_ids"]
        emb_layer = wrapper.model.get_input_embeddings()
        embeds = emb_layer(input_ids).detach()
        embeds.requires_grad_(True)
        inputs = {k:v for k,v in enc.items() if k != "input_ids"}
        out = wrapper.model(inputs_embeds=embeds, **inputs).logits
        score = out[0, int(predicted_label_id)]
        wrapper.model.zero_grad()
        score.backward()
        grads = embeds.grad.detach()[0]
        vals = torch.norm(grads * embeds.detach()[0], dim=-1).detach().cpu().numpy()
        tokens = wrapper.tokenizer.convert_ids_to_tokens(input_ids[0].detach().cpu().tolist())
        scores={}
        for tok, val in zip(tokens, vals):
            clean = tok.replace("##", "").replace("Ġ", "").replace("▁", "").lower().strip(".,!?;:'\"")
            if clean and clean not in ["[cls]", "[sep]", "[pad]", "<s>", "</s>"]:
                scores[clean] = scores.get(clean, 0.0) + float(max(val, 0.0))
        total = sum(scores.values())
        if total > 0:
            scores = {k: float(v/total) for k,v in scores.items()}
        return scores
    except Exception as e:
        print(" Gradient attribution failed:", repr(e))
        return {}


def compute_attribution(wrapper, row, method="occlusion"):
    if method == "gradient":
        return gradient_attribution(wrapper, row, row.get("predicted_label_id", 0))
    if method == "occlusion":
        return occlusion_attribution(wrapper, row, row.get("predicted_label", ""))
    # Future extension: SHAP / Captum IG. Keep occlusion fallback for runnable Colab execution.
    return occlusion_attribution(wrapper, row, row.get("predicted_label", ""))


def attribution_mismatch(row, attribution):
    cues = row.get("expected_cues", [])
    if isinstance(cues, str):
        try: cues = json.loads(cues.replace("'", '"')) if cues.startswith("[") else [cues]
        except Exception: cues = [cues]
    cues = [str(c).lower() for c in (cues or [])]
    if not cues:
        return 0.5, 0.0
    cue_score = 0.0
    for cue in cues:
        cue_parts = simple_tokens(cue)
        for part in cue_parts:
            cue_score += attribution.get(part, 0.0)
    cue_score = min(1.0, float(cue_score))
    am = 1.0 - cue_score
    return float(am), float(cue_score)


def compute_evfs(fail, am, cs, relation):
    if int(fail) == 0:
        return 0.0
    if relation == "directional":
        return float(am * (1.0 - cs))
    return float(am * cs)


def infer_failure_reason(row, attribution):
    if int(row.get("fail_status", 0)) == 0:
        return "Passed: model prediction matched the expected behavioral label."
    cap = row.get("capability", "")
    cs = float(row.get("CS", 0.0))
    am = float(row.get("AM", 0.0))
    top_tokens = sorted(attribution.items(), key=lambda x: x[1], reverse=True)[:5]
    top_str = ", ".join([f"{t}:{v:.2f}" for t,v in top_tokens])
    if cap == "negation":
        return f"Negation failure: expected negation cues received insufficient attribution (AM={am:.2f}) and directional CS={cs:.2f}; top tokens: {top_str}."
    if cap == "temporal_sentiment":
        return f"Temporal failure: model likely focused on earlier or misleading sentiment cues instead of final/current cue (AM={am:.2f}, CS={cs:.2f}); top tokens: {top_str}."
    if cap == "entity_invariance":
        return f"Entity sensitivity: prediction changed or failed under irrelevant entity/name variation; invariance CS={cs:.2f}; top tokens: {top_str}."
    if cap == "robustness":
        return f"Robustness failure: model was sensitive to typo/noise or surface perturbation; invariance CS={cs:.2f}; top tokens: {top_str}."
    if cap == "lexical_overlap":
        return f"Lexical-overlap failure: model likely over-relied on shared words rather than semantic roles/logical relation; AM={am:.2f}, CS={cs:.2f}; top tokens: {top_str}."
    if cap == "contradiction":
        return f"Contradiction failure: model likely underweighted contradiction markers such as not/no/nobody; AM={am:.2f}, CS={cs:.2f}; top tokens: {top_str}."
    if cap == "word_order_sensitivity":
        return f"Word-order sensitivity failure: model likely over-relied on overlapping terms and missed role/order reversal; AM={am:.2f}, CS={cs:.2f}; top tokens: {top_str}."
    return f"General diagnostic failure: attribution/counterfactual evidence indicates mismatch (AM={am:.2f}, CS={cs:.2f}); top tokens: {top_str}."

print(" Explanation and diagnostic utilities ready")




## 8. Run Full X-CheckList Experiment Across Dataset × Model Pairs


In [ ]:
print(" Running X-CheckList across all available dataset × model pairs...")
print("Configured datasets:", CONFIG["datasets"])
print("Configured models:", CONFIG["models"])

all_results=[]
run_log=[]

for dataset_key in CONFIG["datasets"]:
    if dataset_key not in loaded_datasets:
        print(f"Skipping {dataset_key}: dataset not loaded")
        continue
    task = DATASET_SPECS[dataset_key]["task"]
    tests_subset = test_cases_df[test_cases_df["dataset"] == dataset_key].reset_index(drop=True)
    for model_key in CONFIG["models"]:
        model_name = MODEL_REGISTRY.get(dataset_key, {}).get(model_key)
        if not model_name:
            print(f" No checkpoint configured for {dataset_key}/{model_key}; skipping")
            continue
        wrapper = None
        try:
            wrapper = TargetModelWrapper(model_key, dataset_key, model_name, task, device=device, max_length=CONFIG["max_length"])
            pred_df = run_predictions_for_combo(dataset_key, model_key, wrapper, tests_subset)
            # Explain failed cases up to cap. Non-failed cases receive neutral/no attribution diagnostics.
            pred_df["AM"] = 0.0
            pred_df["cue_attribution_ratio"] = 0.0
            pred_df["EVFS"] = 0.0
            pred_df["explanation_method"] = "not_explained"
            pred_df["top_attribution_tokens"] = "[]"
            pred_df["failure_reason"] = "Passed or not explained."
            failed_indices = pred_df.index[pred_df["fail_status"] == 1].tolist()
            max_exp = CONFIG["max_explain_cases_per_combo"]
            if max_exp is not None:
                failed_indices = failed_indices[:max_exp]
            print(f" Explaining failed cases for {dataset_key}/{model_key}: {len(failed_indices)}")
            for idx in tqdm(failed_indices, desc=f"Explain {dataset_key}-{model_key}"):
                row = pred_df.loc[idx].to_dict()
                method = CONFIG["explanation_methods"][0]
                if "gradient" in CONFIG["explanation_methods"]:
                    method = "gradient"
                attr = compute_attribution(wrapper, row, method=method)
                if not attr and "occlusion" in CONFIG["explanation_methods"]:
                    method = "occlusion"
                    attr = compute_attribution(wrapper, row, method="occlusion")
                am, cue_ratio = attribution_mismatch(row, attr)
                evfs = compute_evfs(row["fail_status"], am, row["CS"], row["relation"])
                pred_df.at[idx, "AM"] = am
                pred_df.at[idx, "cue_attribution_ratio"] = cue_ratio
                pred_df.at[idx, "EVFS"] = evfs
                pred_df.at[idx, "explanation_method"] = method
                top_tokens = sorted(attr.items(), key=lambda x: x[1], reverse=True)[:10]
                pred_df.at[idx, "top_attribution_tokens"] = json.dumps(top_tokens)
                row["AM"] = am
                row["EVFS"] = evfs
                pred_df.at[idx, "failure_reason"] = infer_failure_reason(row, attr)
            all_results.append(pred_df)
            run_log.append({"dataset": dataset_key, "model": model_key, "checkpoint": model_name, "status": "success", "rows": len(pred_df), "failed": int(pred_df["fail_status"].sum())})
            print(f" Completed {dataset_key}/{model_key}: rows={len(pred_df)}, failure_rate={pred_df['fail_status'].mean():.3f}, mean_EVFS={pred_df['EVFS'].mean():.3f}")
        except Exception as e:
            print(f" Failed run {dataset_key}/{model_key}: {repr(e)}")
            traceback.print_exc(limit=1)
            run_log.append({"dataset": dataset_key, "model": model_key, "checkpoint": model_name, "status": "failed", "error": repr(e)})
        finally:
            if wrapper is not None:
                wrapper.cleanup()

if len(all_results) == 0:
    raise RuntimeError("No successful experiment runs. Check model checkpoint names, internet access, and GPU/CPU memory.")

results_df = pd.concat(all_results, ignore_index=True)
run_log_df = pd.DataFrame(run_log)

print(" X-CheckList experiment completed")
print("Results shape:", results_df.shape)
print("Run log:")
display(run_log_df)
print("Result preview:")
display(results_df.head())

results_path = output_dir / "xchecklist_instance_report.csv"
run_log_path = output_dir / "xchecklist_run_log.csv"
results_df.to_csv(results_path, index=False)
run_log_df.to_csv(run_log_path, index=False)
generated_files += [str(results_path), str(run_log_path)]
print("Saved:", results_path)
print("Saved:", run_log_path)




## 9. Aggregate Diagnostic Reports


In [ ]:
print(" Creating aggregate and failed-case diagnostic reports...")

def mode_reason(series):
    vals = [x for x in series if isinstance(x, str) and x.strip()]
    if not vals:
        return "No reason available"
    return Counter(vals).most_common(1)[0][0]

aggregate_report = (
    results_df.groupby(["dataset", "task", "model", "capability", "relation"])
    .agg(total_tests=("test_id", "count"), failed_tests=("fail_status", "sum"),
         failure_rate=("fail_status", "mean"), mean_confidence=("confidence", "mean"),
         mean_AM=("AM", "mean"), mean_CS=("CS", "mean"), mean_EVFS=("EVFS", "mean"),
         mean_cue_attribution_ratio=("cue_attribution_ratio", "mean"),
         main_failure_reason=("failure_reason", mode_reason))
    .reset_index()
    .sort_values(["failure_rate", "mean_EVFS"], ascending=False)
)

failed_cases_report = results_df[results_df["fail_status"] == 1].copy().sort_values("EVFS", ascending=False)

aggregate_path = output_dir / "xchecklist_aggregate_report.csv"
failed_path = output_dir / "xchecklist_failed_cases.csv"
aggregate_report.to_csv(aggregate_path, index=False)
failed_cases_report.to_csv(failed_path, index=False)
generated_files += [str(aggregate_path), str(failed_path)]

print("Aggregate reports created")
print("Aggregate shape:", aggregate_report.shape)
display(aggregate_report.head(20))
print("Failed cases shape:", failed_cases_report.shape)
display(failed_cases_report.head(10))
print("Saved:", aggregate_path)
print("Saved:", failed_path)




## 10. Counterfactual Quality Control Report


In [ ]:
print(" Creating counterfactual quality control report...")

def token_set(s): return set(simple_tokens(s))

def jaccard(a,b):
    a,b=token_set(a),token_set(b)
    if not a and not b: return 1.0
    return len(a & b) / max(1, len(a | b))

def quality_row(row):
    orig = row_text_for_explanation(row)
    if pd.notna(row.get("cf_sentence1", np.nan)) and str(row.get("cf_sentence1")) != "nan":
        cf = str(row.get("cf_sentence1")) + " [SEP] " + str(row.get("cf_sentence2"))
    else:
        cf = str(row.get("counterfactual_text", ""))
    sim = jaccard(orig, cf)
    # Heuristic automatic QC scores; final paper should manually validate a sample.
    minimality = float(min(1.0, max(0.0, sim)))
    grammaticality = 1.0 if len(simple_tokens(cf)) >= 3 else 0.0
    label_validity = 1.0 if row.get("counterfactual_expected_label") in TASK_LABELS[row.get("task")] else 0.0
    semantic_control = 1.0 if row.get("relation") in ["directional", "invariance"] else 0.0
    return {
        "test_id": row["test_id"], "dataset": row["dataset"], "task": row["task"], "capability": row["capability"],
        "original": orig, "counterfactual": cf, "relation": row["relation"],
        "minimality_score": minimality, "grammaticality_score": grammaticality,
        "label_validity_score": label_validity, "semantic_control_score": semantic_control,
        "auto_quality_mean": np.mean([minimality, grammaticality, label_validity, semantic_control]),
        "human_validated": False, "human_notes": ""
    }

counterfactual_quality_report = pd.DataFrame([quality_row(r) for _, r in test_cases_df.iterrows()])
cfq_path = output_dir / "counterfactual_quality_report.csv"
counterfactual_quality_report.to_csv(cfq_path, index=False)
generated_files.append(str(cfq_path))

print("Counterfactual quality report created")
display(counterfactual_quality_report.head(10))
print("Saved:", cfq_path)




## 11. Baseline Comparison and Ablation Study


In [ ]:
print("Creating baseline comparison using Diagnostic Coverage Score (DCS)...")

def diagnostic_coverage_score(
    failure_detection=True,
    attribution=False,
    counterfactual=False,
    evfs=False,
    failure_reason=0.0
):
    """
    Compute Diagnostic Coverage Score (DCS).

    DCS = (I_FD + I_AT + I_CF + I_EVFS + I_FR) / 5

    failure_reason:
        0.0 = no failure reason
        0.5 = partial failure reason
        1.0 = full human-readable failure reason
    """
    return (
        int(bool(failure_detection))
        + int(bool(attribution))
        + int(bool(counterfactual))
        + int(bool(evfs))
        + float(failure_reason)
    ) / 5.0


baseline_rows = []

for (dataset, task, model, capability), g in results_df.groupby(["dataset", "task", "model", "capability"]):
    failure_rate = g["fail_status"].mean()
    mean_am = g["AM"].mean()
    mean_cs = g["CS"].mean()
    mean_evfs = g["EVFS"].mean()

    # Method-level DCS values
    # Standard CheckList: failure detection only
    standard_dcs = diagnostic_coverage_score(
        failure_detection=True,
        attribution=False,
        counterfactual=False,
        evfs=False,
        failure_reason=0.0
    )

    # CheckList + Attribution: failure detection + attribution + partial reason
    attribution_dcs = diagnostic_coverage_score(
        failure_detection=True,
        attribution=True,
        counterfactual=False,
        evfs=False,
        failure_reason=0.5
    )

    # CheckList + Counterfactual: failure detection + counterfactual + partial reason
    counterfactual_dcs = diagnostic_coverage_score(
        failure_detection=True,
        attribution=False,
        counterfactual=True,
        evfs=False,
        failure_reason=0.5
    )

    # X-CheckList: all diagnostic components
    xchecklist_dcs = diagnostic_coverage_score(
        failure_detection=True,
        attribution=True,
        counterfactual=True,
        evfs=True,
        failure_reason=1.0
    )

    baseline_rows.extend([
        {
            "method": "Standard CheckList",
            "dataset": dataset,
            "task": task,
            "model": model,
            "capability": capability,
            "failure_detection": True,
            "attribution_evidence": False,
            "counterfactual_validation": False,
            "evfs_available": False,
            "failure_reason": "no",
            "failure_rate": failure_rate,
            "mean_AM": np.nan,
            "mean_CS": np.nan,
            "mean_EVFS": np.nan,
            "diagnostic_coverage_score": standard_dcs
        },
        {
            "method": "CheckList + Attribution",
            "dataset": dataset,
            "task": task,
            "model": model,
            "capability": capability,
            "failure_detection": True,
            "attribution_evidence": True,
            "counterfactual_validation": False,
            "evfs_available": False,
            "failure_reason": "partial",
            "failure_rate": failure_rate,
            "mean_AM": mean_am,
            "mean_CS": np.nan,
            "mean_EVFS": np.nan,
            "diagnostic_coverage_score": attribution_dcs
        },
        {
            "method": "CheckList + Counterfactual",
            "dataset": dataset,
            "task": task,
            "model": model,
            "capability": capability,
            "failure_detection": True,
            "attribution_evidence": False,
            "counterfactual_validation": True,
            "evfs_available": False,
            "failure_reason": "partial",
            "failure_rate": failure_rate,
            "mean_AM": np.nan,
            "mean_CS": mean_cs,
            "mean_EVFS": np.nan,
            "diagnostic_coverage_score": counterfactual_dcs
        },
        {
            "method": "X-CheckList",
            "dataset": dataset,
            "task": task,
            "model": model,
            "capability": capability,
            "failure_detection": True,
            "attribution_evidence": True,
            "counterfactual_validation": True,
            "evfs_available": True,
            "failure_reason": "yes",
            "failure_rate": failure_rate,
            "mean_AM": mean_am,
            "mean_CS": mean_cs,
            "mean_EVFS": mean_evfs,
            "diagnostic_coverage_score": xchecklist_dcs
        },
    ])

baseline_comparison_df = pd.DataFrame(baseline_rows)

baseline_path = output_dir / "baseline_comparison_report.csv"
baseline_comparison_df.to_csv(baseline_path, index=False)

generated_files.append(str(baseline_path))

print("Baseline comparison created with Diagnostic Coverage Score")
print("Saved:", baseline_path)

display(baseline_comparison_df.head(12))

print("\nMethod-level baseline summary:")
baseline_summary = (
    baseline_comparison_df
    .groupby("method", as_index=False)
    .agg(
        failure_rate=("failure_rate", "mean"),
        mean_AM=("mean_AM", "mean"),
        mean_CS=("mean_CS", "mean"),
        mean_EVFS=("mean_EVFS", "mean"),
        diagnostic_coverage_score=("diagnostic_coverage_score", "mean")
    )
)

method_order = [
    "Standard CheckList",
    "CheckList + Attribution",
    "CheckList + Counterfactual",
    "X-CheckList"
]

baseline_summary["method"] = pd.Categorical(
    baseline_summary["method"],
    categories=method_order,
    ordered=True
)

baseline_summary = baseline_summary.sort_values("method").reset_index(drop=True)

display(baseline_summary)


print("Creating ablation study report...")

ablation_rows = []

for (dataset, task, model, capability), g in results_df.groupby(["dataset", "task", "model", "capability"]):
    base = {
        "dataset": dataset,
        "task": task,
        "model": model,
        "capability": capability,
        "failure_rate": g["fail_status"].mean(),
        "mean_AM": g["AM"].mean(),
        "mean_CS": g["CS"].mean(),
        "mean_EVFS": g["EVFS"].mean()
    }

    ablation_rows.append({
        **base,
        "variant": "Full X-CheckList",
        "removed_component": "None",
        "diagnostic_completeness": 1.00
    })

    ablation_rows.append({
        **base,
        "variant": "Without Attribution",
        "removed_component": "Attribution/AM",
        "diagnostic_completeness": 0.75,
        "mean_AM": np.nan
    })

    ablation_rows.append({
        **base,
        "variant": "Without Counterfactual Validation",
        "removed_component": "CS",
        "diagnostic_completeness": 0.75,
        "mean_CS": np.nan
    })

    ablation_rows.append({
        **base,
        "variant": "Without EVFS",
        "removed_component": "EVFS",
        "diagnostic_completeness": 0.75,
        "mean_EVFS": np.nan
    })

    ablation_rows.append({
        **base,
        "variant": "Without Failure Reason Rules",
        "removed_component": "Reason inference",
        "diagnostic_completeness": 0.80
    })

ablation_report = pd.DataFrame(ablation_rows)

ablation_path = output_dir / "ablation_report.csv"
ablation_report.to_csv(ablation_path, index=False)

generated_files.append(str(ablation_path))

print(" Ablation report created")
print("Saved:", ablation_path)

display(ablation_report.head(12))



## 12. Human Evaluation Template

This section exports a 150 failed-case human-evaluation template. Annotators fill the score columns, then the next cell **requires** the completed CSV upload and validates it.


In [ ]:
print(" Creating human evaluation ")

# Human evaluation methods and annotators
methods = CONFIG.get("human_eval_methods", ["X-CheckList"])
annotator_ids = CONFIG.get("human_eval_annotator_ids", ["A1", "A2"])
human_eval_sample_size = int(CONFIG.get("human_eval_sample_size", 150))
strata_cols = CONFIG.get("human_eval_strata_cols", ["dataset", "model", "capability"])


def safe_value(row, col, default="N/A"):
    try:
        val = row.get(col, default)
        if pd.isna(val):
            return default
        return val
    except Exception:
        return default


def stratified_sample_df(df, n, strata_cols, seed=42):
    """Balanced fixed-seed sampling over available strata columns.

    Returns up to n rows. If there are fewer than n failed cases, returns all rows.
    """
    df = df.copy().reset_index(drop=True)
    if len(df) <= n:
        return df.sample(frac=1, random_state=seed).reset_index(drop=True)

    use_cols = [c for c in strata_cols if c in df.columns]
    if not use_cols:
        return df.sample(n=n, random_state=seed).reset_index(drop=True)

    rng = np.random.default_rng(seed)
    grouped = [(key, g.copy()) for key, g in df.groupby(use_cols, dropna=False)]
    rng.shuffle(grouped)

    num_groups = len(grouped)
    base_quota = n // num_groups
    remainder = n % num_groups

    sampled_parts = []
    sampled_indices = set()

    for rank, (_, group) in enumerate(grouped):
        quota = base_quota + (1 if rank < remainder else 0)
        if quota <= 0:
            continue
        take = min(quota, len(group))
        part = group.sample(n=take, random_state=seed + rank)
        sampled_parts.append(part)
        sampled_indices.update(part.index.tolist())

    sampled = pd.concat(sampled_parts, ignore_index=False) if sampled_parts else pd.DataFrame(columns=df.columns)

    # Fill any shortfall from rows not already selected.
    if len(sampled) < n:
        remaining = df.drop(index=list(sampled_indices), errors="ignore")
        fill_n = min(n - len(sampled), len(remaining))
        if fill_n > 0:
            fill = remaining.sample(n=fill_n, random_state=seed)
            sampled = pd.concat([sampled, fill], ignore_index=False)

    return sampled.sample(frac=1, random_state=seed).head(n).reset_index(drop=True)


def make_report_text_for_method(row, method):
    """
    Creates method-specific report text for human evaluation.
    """

    dataset = safe_value(row, "dataset")
    model = safe_value(row, "model")
    capability = safe_value(row, "capability")
    relation = safe_value(row, "relation")
    expected = safe_value(row, "expected_label")
    predicted = safe_value(row, "predicted_label")
    input_text = row_text_for_explanation(row)

    cf_predicted = safe_value(row, "cf_predicted_label")
    cf_fail_status = safe_value(row, "cf_fail_status")
    am = float(safe_value(row, "AM", 0.0))
    cs = float(safe_value(row, "CS", 0.0))
    evfs = float(safe_value(row, "EVFS", 0.0))
    cue_ratio = float(safe_value(row, "cue_attribution_ratio", 0.0))
    top_tokens = safe_value(row, "top_attribution_tokens", "[]")
    failure_reason = safe_value(row, "failure_reason", "No diagnostic reason available.")

    base_info = (
        f"Dataset: {dataset}. Model: {model}. Capability: {capability}. "
        f"Relation: {relation}. Input: {input_text}. "
        f"Expected label: {expected}. Predicted label: {predicted}."
    )

    if method == "Standard CheckList":
        return (
            f"Standard CheckList report: {base_info} "
            f"The report identifies this as a failed behavioral test because the predicted label "
            f"does not match the expected label. No attribution evidence, counterfactual validation, "
            f"or detailed failure reason is provided."
        )

    if method == "CheckList + Attribution":
        return (
            f"CheckList + Attribution report: {base_info} "
            f"The report adds token-level attribution evidence. "
            f"Attribution Mismatch Score AM={am:.3f}; cue attribution ratio={cue_ratio:.3f}. "
            f"Top attributed tokens: {top_tokens}. "
            f"This report suggests whether the model focused on expected cues or misleading tokens, "
            f"but it does not validate the explanation using counterfactual behavior."
        )

    if method == "CheckList + Counterfactual":
        return (
            f"CheckList + Counterfactual report: {base_info} "
            f"The counterfactual prediction was {cf_predicted}, with counterfactual fail status={cf_fail_status}. "
            f"Counterfactual sensitivity CS={cs:.3f}. "
            f"This report shows whether the model changed under a controlled counterfactual input, "
            f"but it does not include token-level attribution or a full explanation-validated failure score."
        )

    if method == "X-CheckList":
        return (
            f"X-CheckList report: {base_info} "
            f"Attribution evidence: AM={am:.3f}, cue attribution ratio={cue_ratio:.3f}, "
            f"top attributed tokens={top_tokens}. "
            f"Counterfactual evidence: counterfactual predicted label={cf_predicted}, "
            f"counterfactual fail status={cf_fail_status}, CS={cs:.3f}. "
            f"Explanation-Validated Failure Score EVFS={evfs:.3f}. "
            f"Diagnostic failure reason: {failure_reason}"
        )

    return f"No report available for method: {method}"


failed_source = failed_cases_report.copy()
if len(failed_source) == 0:
    print("No failed cases found. Falling back to results_df sample for template structure.")
    failed_source = results_df.copy()

failed_sample = stratified_sample_df(
    failed_source,
    n=human_eval_sample_size,
    strata_cols=strata_cols,
    seed=CONFIG["random_seed"]
)

# More accurate filename: sample size may be 75, 150, or lower in debug mode.
human_eval_sample_path = output_dir / f"human_eval_failed_case_sample_{len(failed_sample)}.csv"
failed_sample.to_csv(human_eval_sample_path, index=False)
if str(human_eval_sample_path) not in generated_files:
    generated_files.append(str(human_eval_sample_path))

human_rows = []

for _, r in failed_sample.iterrows():
    for method in methods:
        for annotator_id in annotator_ids:

            # IMPORTANT:
            # Only X-CheckList has a diagnostic failure reason.
            # Baseline reports should not inherit this explanation.
            method_failure_reason = (
                r.get("failure_reason", "N/A")
                if method == "X-CheckList"
                else "N/A"
            )

            human_rows.append({
                "annotator_id": annotator_id,
                "method": method,
                "sample_id": r.get("test_id", "N/A"),
                "dataset": r.get("dataset", "N/A"),
                "model": r.get("model", "N/A"),
                "capability": r.get("capability", "N/A"),
                "input_text": row_text_for_explanation(r),
                "expected_label": r.get("expected_label", "N/A"),
                "predicted_label": r.get("predicted_label", "N/A"),
                "failure_reason": method_failure_reason,
                "report_text": make_report_text_for_method(r, method),
                "correctness_1_to_5": np.nan,
                "understandability_1_to_5": np.nan,
                "usefulness_1_to_5": np.nan,
                "actionability_1_to_5": np.nan,
                "comments": "",
            })

human_evaluation_template = pd.DataFrame(human_rows)

human_template_path = output_dir / "human_evaluation_template.csv"
human_evaluation_template.to_csv(human_template_path, index=False)

if str(human_template_path) not in generated_files:
    generated_files.append(str(human_template_path))

print("Human evaluation template created")
print("Failed-case sample size:", len(failed_sample))
print("Template rows to fill:", len(human_evaluation_template))
print("Methods:", methods)
print("Annotators:", annotator_ids)

print("\nFailure reason counts by method:")
print(human_evaluation_template.groupby("method")["failure_reason"].apply(lambda x: (x != "N/A").sum()))

print("\nSample distribution by capability:")
print(failed_sample["capability"].value_counts() if "capability" in failed_sample.columns else "N/A")

print("Saved failed-case sample:", human_eval_sample_path)
print("Saved annotation template:", human_template_path)
display(human_evaluation_template.head())

In [ ]:

print("Uploading and validating completed human evaluation results...")

import shutil
from pathlib import Path
import pandas as pd
import sys

# Required completed file name after human annotation.
REQUIRED_HUMAN_EVAL_FILENAME = "human_evaluation_results.csv"
target_file = output_dir / REQUIRED_HUMAN_EVAL_FILENAME

if "google.colab" in sys.modules:
    from google.colab import files

    print("Upload the completed human evaluation CSV now.")
    print("Required filename:", REQUIRED_HUMAN_EVAL_FILENAME)

    uploaded = files.upload()
    uploaded_names = list(uploaded.keys())

    print("Uploaded files:", uploaded_names)

    
    matching_files = [
        name for name in uploaded_names
        if name.startswith("human_evaluation_results") and name.endswith(".csv")
    ]

    if not matching_files:
        raise FileNotFoundError(
            f"No valid human evaluation CSV uploaded. "
            f"Expected a file like '{REQUIRED_HUMAN_EVAL_FILENAME}', "
            f"but uploaded files were: {uploaded_names}"
        )

    uploaded_name = matching_files[0]
    source_file = Path("/content") / uploaded_name

    print(f"Using uploaded file: {source_file}")

else:
    source_file = Path(REQUIRED_HUMAN_EVAL_FILENAME)

if not source_file.exists():
    raise FileNotFoundError(
        f"Required completed human-evaluation file not found: {source_file}. "
        f"Create/rename the completed annotation file as '{REQUIRED_HUMAN_EVAL_FILENAME}' and rerun this cell."
    )

# Copy uploaded file into the official outputs folder.
shutil.copy(source_file, target_file)

print(" Human evaluation results copied successfully")
print("Saved as:", target_file)

df = pd.read_csv(target_file)

required_cols = [
    "annotator_id",
    "method",
    "sample_id",
    "correctness_1_to_5",
    "understandability_1_to_5",
    "usefulness_1_to_5",
    "actionability_1_to_5"
]

score_cols = [
    "correctness_1_to_5",
    "understandability_1_to_5",
    "usefulness_1_to_5",
    "actionability_1_to_5"
]

missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(
        f"Missing required columns in completed human-evaluation CSV: {missing_cols}"
    )

for col in score_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

missing_scores = int(df[score_cols].isna().sum().sum())
out_of_range = int(((df[score_cols] < 1) | (df[score_cols] > 5)).sum().sum())

print("Rows:", len(df))

if "method" in df.columns:
    print("\nMethods in uploaded human evaluation file:")
    print(df["method"].value_counts())

if "annotator_id" in df.columns:
    print("\nAnnotators:")
    print(df["annotator_id"].value_counts())

print("\nMissing score cells:", missing_scores)
print("Out-of-range score cells:", out_of_range)

if missing_scores > 0:
    raise ValueError(
        "Some human-evaluation score cells are empty. "
        "Fill all 1–5 score columns and rerun this cell."
    )

if out_of_range > 0:
    raise ValueError(
        "Some scores are outside the 1–5 range. Fix them and rerun this cell."
    )

print("\nCompleted human evaluation file is valid and ready for summary.")
display(df.head())

In [ ]:
print("Generating human evaluation summary and comparison...")

def summarize_human_evaluation(completed_csv_path):
    df = pd.read_csv(completed_csv_path)

    metric_cols = [
        "correctness_1_to_5",
        "understandability_1_to_5",
        "usefulness_1_to_5",
        "actionability_1_to_5"
    ]

    required_cols = ["method", "sample_id", "annotator_id"] + metric_cols
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing required columns in human evaluation file: {missing_cols}")

    for col in metric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    missing_scores = int(df[metric_cols].isna().sum().sum())
    out_of_range = int(((df[metric_cols] < 1) | (df[metric_cols] > 5)).sum().sum())

    print("Rows loaded:", len(df))
    print("Methods found:")
    print(df["method"].value_counts())
    print("Missing score cells:", missing_scores)
    print("Out-of-range score cells:", out_of_range)

    if missing_scores > 0:
        raise ValueError("Some human evaluation score cells are empty.")
    if out_of_range > 0:
        raise ValueError("Some human evaluation scores are outside the 1–5 range.")

    # Detailed summary: mean, std, count for each method and metric
    summary = (
        df.groupby("method")[metric_cols]
        .agg(["mean", "std", "count"])
        .round(3)
    )

    # Paper-style comparison table: mean scores only
    comparison_table = (
        df.groupby("method")[metric_cols]
        .mean()
        .round(3)
        .reset_index()
    )

    comparison_table = comparison_table.rename(columns={
        "method": "Method",
        "correctness_1_to_5": "Correctness",
        "understandability_1_to_5": "Understandability",
        "usefulness_1_to_5": "Usefulness",
        "actionability_1_to_5": "Actionability"
    })

    # Overall average score for ranking methods
    comparison_table["Overall Mean"] = comparison_table[
        ["Correctness", "Understandability", "Usefulness", "Actionability"]
    ].mean(axis=1).round(3)

    comparison_table = comparison_table.sort_values(
        by="Overall Mean",
        ascending=False
    ).reset_index(drop=True)

    # Method-wise agreement between two annotators
    agreement_rows = []

    for method, method_df in df.groupby("method"):
        for metric in metric_cols:
            piv = method_df.pivot_table(
                index="sample_id",
                columns="annotator_id",
                values=metric,
                aggfunc="first"
            )

            if piv.shape[1] >= 2:
                annotators = list(piv.columns)
                a1, a2 = annotators[:2]

                valid = piv[[a1, a2]].dropna()

                exact_agreement = (
                    float((valid[a1] == valid[a2]).mean())
                    if len(valid)
                    else np.nan
                )

                within_one = (
                    float((abs(valid[a1] - valid[a2]) <= 1).mean())
                    if len(valid)
                    else np.nan
                )

                agreement_rows.append({
                    "method": method,
                    "metric": metric,
                    "annotator_1": a1,
                    "annotator_2": a2,
                    "pairs": len(valid),
                    "exact_agreement": exact_agreement,
                    "within_1_point_agreement": within_one
                })

    agreement = pd.DataFrame(agreement_rows)

    if not agreement.empty:
        agreement["exact_agreement"] = agreement["exact_agreement"].round(3)
        agreement["within_1_point_agreement"] = agreement["within_1_point_agreement"].round(3)

    return summary, agreement, comparison_table


human_eval_results_path = output_dir / "human_evaluation_results.csv"

if not human_eval_results_path.exists():
    raise FileNotFoundError(
        f"Human evaluation results file not found: {human_eval_results_path}. "
        "Run Cell 28 first and upload the completed human_evaluation_results.csv file."
    )

human_summary, human_agreement, human_comparison_table = summarize_human_evaluation(
    human_eval_results_path
)

print("\nDetailed human evaluation summary:")
display(human_summary)

print("\n Paper-style human evaluation comparison table:")
display(human_comparison_table)

print("\n Human annotator agreement:")
display(human_agreement)

# Save outputs
human_summary_path = output_dir / "human_evaluation_summary.csv"
human_agreement_path = output_dir / "human_evaluation_agreement.csv"
human_comparison_path = output_dir / "paper_human_evaluation_comparison.csv"

human_summary.to_csv(human_summary_path)
human_agreement.to_csv(human_agreement_path, index=False)
human_comparison_table.to_csv(human_comparison_path, index=False)

for path in [human_summary_path, human_agreement_path, human_comparison_path]:
    if str(path) not in generated_files:
        generated_files.append(str(path))

print("\n Saved human evaluation files:")
print("Summary:", human_summary_path)
print("Agreement:", human_agreement_path)
print("Paper-style comparison:", human_comparison_path)



## 13. Statistical Testing


In [ ]:
print(" Running statistical tests for EVFS/failure-rate reliability...")

def bootstrap_ci(values, n_bootstrap=1000, ci=95, seed=42):
    values = np.array(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = [np.mean(rng.choice(values, size=len(values), replace=True)) for _ in range(n_bootstrap)]
    alpha = (100-ci)/2
    return float(np.percentile(means, alpha)), float(np.percentile(means, 100-alpha))

stat_rows=[]
for (dataset, model, capability), g in results_df.groupby(["dataset", "model", "capability"]):
    evfs_low, evfs_high = bootstrap_ci(g["EVFS"], CONFIG["bootstrap_samples"], 95, CONFIG["random_seed"])
    fr_low, fr_high = bootstrap_ci(g["fail_status"], CONFIG["bootstrap_samples"], 95, CONFIG["random_seed"])
    stat_rows.append({"dataset": dataset, "model": model, "capability": capability,
                     "metric": "EVFS", "mean": g["EVFS"].mean(), "ci95_low": evfs_low, "ci95_high": evfs_high})
    stat_rows.append({"dataset": dataset, "model": model, "capability": capability,
                     "metric": "failure_rate", "mean": g["fail_status"].mean(), "ci95_low": fr_low, "ci95_high": fr_high})

# Model-pair Wilcoxon comparisons for EVFS where paired test IDs exist
models = sorted(results_df["model"].unique())
for dataset in sorted(results_df["dataset"].unique()):
    for capability in sorted(results_df[results_df["dataset"] == dataset]["capability"].unique()):
        sub = results_df[(results_df["dataset"] == dataset) & (results_df["capability"] == capability)]
        for i in range(len(models)):
            for j in range(i+1, len(models)):
                m1, m2 = models[i], models[j]
                a = sub[sub["model"] == m1][["test_id", "EVFS"]]
                b = sub[sub["model"] == m2][["test_id", "EVFS"]]
                paired = a.merge(b, on="test_id", suffixes=(f"_{m1}", f"_{m2}"))
                if len(paired) >= 5:
                    try:
                        stat, p = wilcoxon(paired[f"EVFS_{m1}"], paired[f"EVFS_{m2}"])
                    except Exception:
                        stat, p = np.nan, np.nan
                    stat_rows.append({"dataset": dataset, "model": f"{m1}_vs_{m2}", "capability": capability,
                                     "metric": "wilcoxon_evfs", "mean": np.nan, "ci95_low": np.nan, "ci95_high": np.nan,
                                     "statistic": stat, "p_value": p})

statistical_test_report = pd.DataFrame(stat_rows)
stat_path = output_dir / "statistical_test_report.csv"
statistical_test_report.to_csv(stat_path, index=False)
generated_files.append(str(stat_path))

print(" Statistical report created")
display(statistical_test_report.head(20))
print("Saved:", stat_path)




## 14. Figures


In [ ]:
print(" Generating publication-style figures...")

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# Global plotting style
# ---------------------------------------------------------------------

sns.set_theme(style="whitegrid", context="paper", font_scale=1.10)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "legend.title_fontsize": 10,
    "axes.titleweight": "bold"
})


def finalize_plot(fig_path, rect=None):
    """
    Save figure safely without cutting title, subtitle, labels, or annotations.
    """
    if rect is None:
        plt.tight_layout()
    else:
        plt.tight_layout(rect=rect)

    plt.savefig(fig_path, dpi=300, bbox_inches="tight", pad_inches=0.15)
    plt.show()
    generated_files.append(str(fig_path))


def annotate_bars(ax, fmt="{:.2f}", rotation=0, fontsize=9):
    """
    Add value labels above vertical bars.
    """
    for p in ax.patches:
        height = p.get_height()
        if height is not None and not np.isnan(height):
            ax.annotate(
                fmt.format(height),
                (p.get_x() + p.get_width() / 2., height),
                ha="center",
                va="bottom",
                fontsize=fontsize,
                rotation=rotation,
                xytext=(0, 5),
                textcoords="offset points"
            )


def clean_label(x):
    return str(x).replace("_", " ").title()


# ---------------------------------------------------------------------
# 1. Failure rate by capability
# ---------------------------------------------------------------------

plt.figure(figsize=(13, 6.2))

fr_cap = (
    results_df.groupby("capability")["fail_status"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

fr_cap["capability_label"] = fr_cap["capability"].apply(clean_label)

ax = sns.barplot(
    data=fr_cap,
    x="capability_label",
    y="fail_status",
    palette="Blues_r",
    edgecolor="black",
    linewidth=0.6
)

plt.title("Failure Rate by Behavioral Capability", pad=18)
plt.xlabel("Capability", labelpad=10)
plt.ylabel("Failure Rate", labelpad=8)
plt.xticks(rotation=30, ha="right")

ymax = max(fr_cap["fail_status"].max() * 1.25, 0.08)
plt.ylim(0, ymax)

annotate_bars(ax, fmt="{:.2f}", fontsize=9)

sns.despine()
fig1 = figures_dir / "failure_rate_by_capability.png"
finalize_plot(fig1)


# ---------------------------------------------------------------------
# 2. Mean EVFS by capability
# ---------------------------------------------------------------------

plt.figure(figsize=(13, 6.2))

evfs_cap = (
    results_df.groupby("capability")["EVFS"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

evfs_cap["capability_label"] = evfs_cap["capability"].apply(clean_label)

ax = sns.barplot(
    data=evfs_cap,
    x="capability_label",
    y="EVFS",
    palette="Greens_r",
    edgecolor="black",
    linewidth=0.6
)

plt.title("Mean Explanation-Validated Failure Score by Capability", pad=18)
plt.xlabel("Capability", labelpad=10)
plt.ylabel("Mean EVFS", labelpad=8)
plt.xticks(rotation=30, ha="right")

ymax = max(evfs_cap["EVFS"].max() * 1.25, 0.08)
plt.ylim(0, ymax)

annotate_bars(ax, fmt="{:.2f}", fontsize=9)

sns.despine()
fig2 = figures_dir / "mean_evfs_by_capability.png"
finalize_plot(fig2)


# ---------------------------------------------------------------------
# 3. Model × dataset failure heatmap
# ---------------------------------------------------------------------

pivot = (
    results_df.groupby(["dataset", "model"])["fail_status"]
    .mean()
    .reset_index()
    .pivot(index="dataset", columns="model", values="fail_status")
)

dataset_order = [d for d in ["sst2", "imdb", "mnli", "snli", "qqp", "mrpc"] if d in pivot.index]
model_order = [m for m in ["distilbert", "bert", "roberta"] if m in pivot.columns]

pivot = pivot.loc[dataset_order, model_order]

plt.figure(figsize=(9.2, 6.0))

ax = sns.heatmap(
    pivot,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.7,
    linecolor="white",
    cbar_kws={"label": "Failure Rate", "shrink": 0.85},
    annot_kws={"fontsize": 10}
)

plt.title("Failure Rate Heatmap by Dataset and Model", pad=18)
plt.xlabel("Model", labelpad=10)
plt.ylabel("Dataset", labelpad=8)

ax.set_xticklabels([clean_label(x.get_text()) for x in ax.get_xticklabels()], rotation=0)
ax.set_yticklabels([x.get_text().upper() for x in ax.get_yticklabels()], rotation=0)

fig3 = figures_dir / "model_dataset_failure_heatmap.png"
finalize_plot(fig3)


# ---------------------------------------------------------------------
# 4. Top diagnostic capabilities ranked by mean EVFS
# ---------------------------------------------------------------------

plt.figure(figsize=(11.5, 6.8))

rank_df = results_df.copy()

rank_df = rank_df[
    (rank_df["fail_status"] == 1) |
    (rank_df["EVFS"] > 0) |
    (rank_df["AM"] > 0)
].copy()

if len(rank_df) == 0:
    print("⚠️ No failed/explained cases found. Using all results as fallback.")
    rank_df = results_df.copy()

rank_summary = (
    rank_df.groupby("capability")
    .agg(
        mean_EVFS=("EVFS", "mean"),
        mean_AM=("AM", "mean"),
        mean_CS=("CS", "mean"),
        cases=("test_id", "count"),
        failure_rate=("fail_status", "mean")
    )
    .reset_index()
    .sort_values("mean_EVFS", ascending=False)
)

rank_summary["capability_label"] = rank_summary["capability"].apply(clean_label)

ax = sns.barplot(
    data=rank_summary,
    x="mean_EVFS",
    y="capability_label",
    palette="Greens_r",
    edgecolor="black",
    linewidth=0.6
)

plt.title("Top Diagnostic Capabilities Ranked by Mean EVFS", pad=18)
plt.xlabel("Mean Explanation-Validated Failure Score", labelpad=10)
plt.ylabel("Capability", labelpad=8)

xmax = max(rank_summary["mean_EVFS"].max() * 1.45, 0.10)
plt.xlim(0, xmax)

for i, row in rank_summary.reset_index(drop=True).iterrows():
    ax.text(
        row["mean_EVFS"] + (xmax * 0.015),
        i,
        f"AM={row['mean_AM']:.2f}, CS={row['mean_CS']:.2f}, n={int(row['cases'])}",
        va="center",
        fontsize=9
    )

sns.despine()
fig4 = figures_dir / "top_diagnostic_capabilities_evfs.png"
finalize_plot(fig4)


# ---------------------------------------------------------------------
# 5. Baseline comparison by Diagnostic Coverage Score
# ---------------------------------------------------------------------

if "diagnostic_coverage_score" not in baseline_comparison_df.columns:
    if "diagnostic_evidence_score" in baseline_comparison_df.columns:
        print(" diagnostic_coverage_score not found. Using diagnostic_evidence_score temporarily.")
        baseline_comparison_df["diagnostic_coverage_score"] = baseline_comparison_df["diagnostic_evidence_score"]
    else:
        raise ValueError(
            "Neither diagnostic_coverage_score nor diagnostic_evidence_score was found in baseline_comparison_df."
        )

method_order = [
    "Standard CheckList",
    "CheckList + Attribution",
    "CheckList + Counterfactual",
    "X-CheckList"
]

base_sum = (
    baseline_comparison_df
    .groupby("method", as_index=False)
    .agg(
        failure_rate=("failure_rate", "mean"),
        diagnostic_coverage_score=("diagnostic_coverage_score", "mean")
    )
)

base_sum["method"] = pd.Categorical(
    base_sum["method"],
    categories=method_order,
    ordered=True
)

base_sum = base_sum.sort_values("method").reset_index(drop=True)
base_sum["method_label"] = base_sum["method"].astype(str)

fig, ax = plt.subplots(figsize=(12.5, 6.8))

sns.barplot(
    data=base_sum,
    x="method_label",
    y="diagnostic_coverage_score",
    palette="magma",
    edgecolor="black",
    linewidth=0.6,
    ax=ax
)

ax.set_title(
    "Baseline Comparison by Diagnostic Coverage Score",
    pad=34,
    fontsize=15,
    fontweight="bold"
)

fig.text(
    0.5,
    0.925,
    "DCS measures available diagnostic components, not predictive performance",
    ha="center",
    va="center",
    fontsize=9
)

ax.set_xlabel("Method", labelpad=12)
ax.set_ylabel("Diagnostic Coverage Score", labelpad=8)
ax.set_ylim(0, 1.18)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=22,
    ha="right"
)

annotate_bars(ax, fmt="{:.2f}", fontsize=10)

sns.despine()
fig5 = figures_dir / "baseline_comparison.png"
finalize_plot(fig5, rect=[0, 0.06, 1, 0.88])


print("Figures generated")
print("Saved figures:")
for f in [fig1, fig2, fig3, fig4, fig5]:
    print(f)



## 15. Export Config and Final Summary


In [ ]:
print("Exporting final config, file manifest, and summary...")

config_path = output_dir / "xchecklist_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)
generated_files.append(str(config_path))

manifest_path = output_dir / "generated_files_manifest.txt"
with open(manifest_path, "w", encoding="utf-8") as f:
    for path in generated_files:
        f.write(str(path) + "\n")
generated_files.append(str(manifest_path))

summary = {
    "run_mode": RUN_MODE,
    "datasets_configured": CONFIG["datasets"],
    "models_configured": CONFIG["models"],
    "successful_runs": int((run_log_df["status"] == "success").sum()) if "run_log_df" in globals() else 0,
    "total_test_cases": int(len(test_cases_df)),
    "total_result_rows": int(len(results_df)),
    "total_failed_cases": int(results_df["fail_status"].sum()),
    "overall_failure_rate": float(results_df["fail_status"].mean()),
    "mean_AM": float(results_df["AM"].mean()),
    "mean_CS": float(results_df["CS"].mean()),
    "mean_EVFS": float(results_df["EVFS"].mean()),
}
summary_path = output_dir / "xchecklist_q1_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
generated_files.append(str(summary_path))

print("X-CheckList  completed")
print(json.dumps(summary, indent=2))
print("\nMain reports:")
for p in [
    "xchecklist_instance_report.csv", "xchecklist_aggregate_report.csv", "xchecklist_failed_cases.csv",
    "counterfactual_quality_report.csv", "baseline_comparison_report.csv", "ablation_report.csv",
    "human_evaluation_template.csv", "statistical_test_report.csv"
]:
    print(output_dir / p)

print("\nAll generated files:")
for path in generated_files:
    print(path)




## 16. Optional: Save Outputs to Google Drive


In [ ]:
print("\: Saving outputs to Google Drive... =====")
if CONFIG.get("save_to_google_drive", False):
    print(" Saving outputs to Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    drive_dir = Path(CONFIG["google_drive_output_dir"])
    drive_dir.mkdir(parents=True, exist_ok=True)
    for item in output_dir.iterdir():
        dest = drive_dir / item.name
        if item.is_dir():
            if dest.exists(): shutil.rmtree(dest)
            shutil.copytree(item, dest)
        else:
            shutil.copy2(item, dest)
    print(" Outputs copied to:", drive_dir)
else:
    print("Google Drive saving disabled. To enable: set CONFIG['save_to_google_drive'] = True and rerun this cell.")
